**Filtering, Literal & Casting**

**1. Filtering (filter() / where())**

filter() and where() are identical — both filter rows based on a condition. where() is an alias for filter(), kept for SQL familiarity. we can use either one.

**Using Python's and / or keywords instead of & / | will throw an error. Always use & for AND and | for OR in PySpark column conditions. Always wrap each condition in parentheses when combining them.

**2.Literal**

lit():creates a Column from a constant value

we use it when you want to add a column where every row has the same value 

**3.Casting**

cast():converts a column from one data type to another

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *

In [7]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-6")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


**Task 1**

Filter orders.csv to show only orders where unit_price is greater than 200 AND status is Delivered. How many rows match? Then filter for orders where region is either East OR West.

In [14]:
orders_df=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv",
header=True,inferSchema=True)
print(f"Total Orders: {orders_df.count()}")
filtered_orders=orders_df.filter((F.col("unit_price")>200) & (F.col("status")=="Delivered"))
print(f"High-value Delivered Orders: {filtered_orders.count()}")
filtered_by_region=orders_df.filter((F.col("region")=="East") | (F.col("region")=="West"))
print(f"Orders from East and West Regions: {filtered_by_region.count()}")


Total Orders: 100


High-value Delivered Orders: 25


Orders from East and West Regions: 63


**Task 2**

Using isin(), filter orders where payment_method is either Credit Card or PayPal. Then use between() to filter orders where unit_price is between 50 and 300.

In [17]:
filtered_by_payment=orders_df.where((F.col("payment_method").isin(["Credit Card","PayPal"])))
filtered_by_payment.show(5)
filtered_by_price=orders_df.filter((F.col("unit_price") >= 50) & (F.col("unit_price") <= 500))
filtered_by_price.show(5)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|           0|Delivered|   Credit Card|   West|
|   O0006|       C006|      P008|2023-01-18|       1|    199.99|          10|Delivered|   Credit Card|   East|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
o

**Task 3**

Add three constant columns to orders.csv using lit(): data_source = "DEH_CHALLENGE", version = 1, is_active = True. Show order_id and all three new columns.

orders_df.lit()

**Task 4**

Read orders.csv without a schema (everything comes in as string). Use cast() to convert quantity to integer, unit_price to double, and order_date to date. Print the schema before and after to confirm the types changed.